# Нарезка данных на train/val/test с правильной стратификацией

Разбиение исходных данных (training.csv) на три группы (60/20/20) с учётом:
- Каждый клиент попадает только в одну группу
- Fraud rate совпадает на уровне клиентов
- Fraud rate совпадает на уровне транзакций

## 1. Импорт библиотек

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## 2. Загрузка и исследование данных

In [ ]:
# Загружаем исходные данные
df = pd.read_csv('full.csv')

print(f"Размер датасета: {df.shape}")
print(f"\nКолонки: {df.columns.tolist()}")
print(f"\nПервые несколько строк:")
print(df.head())
print(f"\nТипы данных:")
print(df.dtypes)
print(f"\nПропущенные значения:")
print(df.isnull().sum())

Размер датасета: (95662, 16)

Колонки: ['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId', 'CurrencyCode', 'CountryCode', 'ProviderId', 'ProductId', 'ProductCategory', 'ChannelId', 'Amount', 'Value', 'TransactionStartTime', 'PricingStrategy', 'FraudResult']

Первые несколько строк:
         TransactionId         BatchId       AccountId       SubscriptionId  \
0  TransactionId_76871   BatchId_36123  AccountId_3957   SubscriptionId_887   
1  TransactionId_73770   BatchId_15642  AccountId_4841  SubscriptionId_3829   
2  TransactionId_26203   BatchId_53941  AccountId_4229   SubscriptionId_222   
3    TransactionId_380  BatchId_102363   AccountId_648  SubscriptionId_2185   
4  TransactionId_28195   BatchId_38780  AccountId_4841  SubscriptionId_3829   

        CustomerId CurrencyCode  CountryCode    ProviderId     ProductId  \
0  CustomerId_4406          UGX          256  ProviderId_6  ProductId_10   
1  CustomerId_4406          UGX          256  ProviderId_4   Product

## 3. Очистка данных

In [15]:
# НЕ удаляем колонки - работаем с исходными данными
print(f"Используем все колонки: {df.columns.tolist()}")

Используем все колонки: ['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId', 'CurrencyCode', 'CountryCode', 'ProviderId', 'ProductId', 'ProductCategory', 'ChannelId', 'Amount', 'Value', 'TransactionStartTime', 'PricingStrategy', 'FraudResult']


## 4. Анализ для стратификации

In [16]:
# Анализируем данные по клиентам и мошенничеству
print("=" * 60)
print("ИСХОДНЫЕ СТАТИСТИКИ")
print("=" * 60)

print(f"\nОбщие данные:")
print(f"Всего транзакций: {len(df)}")
print(f"Уникальных клиентов: {df['CustomerId'].nunique()}")
print(f"Fraud rate (транзакции): {df['FraudResult'].mean():.4f}")

# Группируем по клиентам для анализа
customer_stats = df.groupby('CustomerId').agg({
    'FraudResult': ['sum', 'count', 'mean']  # sum=кол-во фрод, count=всего транзакций, mean=fraud rate
}).reset_index()

customer_stats.columns = ['CustomerId', 'fraud_count', 'total_transactions', 'fraud_rate']

print(f"\nПо клиентам:")
print(f"Клиентов с мошенничеством: {(customer_stats['fraud_count'] > 0).sum()}")
print(f"Средний fraud rate по клиентам: {customer_stats['fraud_rate'].mean():.4f}")
print(f"Распределение fraud_rate:")
print(customer_stats['fraud_rate'].describe())

print(f"\nРаспределение числа транзакций на клиента:")
print(customer_stats['total_transactions'].describe())

ИСХОДНЫЕ СТАТИСТИКИ

Общие данные:
Всего транзакций: 95662
Уникальных клиентов: 3742
Fraud rate (транзакции): 0.0020

По клиентам:
Клиентов с мошенничеством: 54
Средний fraud rate по клиентам: 0.0056
Распределение fraud_rate:
count    3742.000000
mean        0.005556
std         0.064539
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: fraud_rate, dtype: float64

Распределение числа транзакций на клиента:
count    3742.000000
mean       25.564404
std        96.929602
min         1.000000
25%         2.000000
50%         7.000000
75%        20.000000
max      4091.000000
Name: total_transactions, dtype: float64


## 5. Стратифицированное разбиение клиентов на 60/20/20

In [17]:
"""
ФИНАЛЬНАЯ ЛОГИКА РАЗБИЕНИЯ - СОВПАДЕНИЕ ПО ТРАНЗАКЦИЯМ:
1. Стратификация по QUANTILE fraud_count (абсолютное число фродов)
2. + Стратификация по fraud_rate (вероятность фрода)
3. Это гарантирует, что фроды распределяются пропорционально

Почему это работает:
- Количество фродов в каждом расщепе должно быть пропорционально ~60/20/20
- Это автоматически даст одинаковый fraud_rate по транзакциям
"""

# КЛЮЧЕВОЙ МОМЕНТ: стратифицируем по КОЛИЧЕСТВУ фродов!
# Используем quantile для fraud_count - это разделит клиентов на группы с одинаковым числом фродов
customer_stats['fraud_count_quantile'] = pd.qcut(
    customer_stats['fraud_count'], 
    q=10,  # 10 групп по числу фродов
    labels=False,
    duplicates='drop'
)

# Также стратифицируем по fraud_rate для лучшего баланса
customer_stats['fraud_rate_quantile'] = pd.qcut(
    customer_stats['fraud_rate'], 
    q=5,  # 5 групп по вероятности фрода
    labels=False,
    duplicates='drop'
)

# Комбинированная страта - это даст ~50 уникальных страт
customer_stats['combined_strata'] = (
    customer_stats['fraud_count_quantile'].astype(int) * 100 + 
    customer_stats['fraud_rate_quantile'].astype(int)
)

print("Распределение клиентов по количеству фродов (fraud_count quantile):")
print(customer_stats['fraud_count_quantile'].value_counts().sort_index())

print("\nРаспределение по combined stratification:")
strata_dist = customer_stats['combined_strata'].value_counts().sort_index()
print(f"Всего уникальных страт: {len(strata_dist)}")

# Первый split: train (60%) vs temp (40%)
train_customers, temp_customers = train_test_split(
    customer_stats,
    test_size=0.4,
    stratify=customer_stats['combined_strata'],
    random_state=42
)

# Второй split: val (20%) vs test (20%) из оставшихся
val_customers, test_customers = train_test_split(
    temp_customers,
    test_size=0.5,
    stratify=temp_customers['combined_strata'],
    random_state=42
)

print(f"\n{'='*60}")
print("РАЗМЕР РАСЩЕПОВ (по клиентам):")
print(f"{'='*60}")
print(f"Train клиентов: {len(train_customers)} ({len(train_customers)/len(customer_stats)*100:.1f}%)")
print(f"Val клиентов:   {len(val_customers)} ({len(val_customers)/len(customer_stats)*100:.1f}%)")
print(f"Test клиентов:  {len(test_customers)} ({len(test_customers)/len(customer_stats)*100:.1f}%)")

# Проверяем, нет ли пересечений
train_ids = set(train_customers['CustomerId'])
val_ids = set(val_customers['CustomerId'])
test_ids = set(test_customers['CustomerId'])

overlap_train_val = len(train_ids & val_ids)
overlap_train_test = len(train_ids & test_ids)
overlap_val_test = len(val_ids & test_ids)

print(f"\nПересечения клиентов:")
print(f"Train ∩ Val:  {overlap_train_val} {'✓ OK' if overlap_train_val == 0 else '✗ ПРОБЛЕМА!'}")
print(f"Train ∩ Test: {overlap_train_test} {'✓ OK' if overlap_train_test == 0 else '✗ ПРОБЛЕМА!'}")
print(f"Val ∩ Test:   {overlap_val_test} {'✓ OK' if overlap_val_test == 0 else '✗ ПРОБЛЕМА!'}")

# Проверка баланса параметров
print(f"\n{'='*60}")
print("ПРОВЕРКА БАЛАНСА ПО КЛИЕНТАМ:")
print(f"{'='*60}")

print(f"\nОбщее число фродов (важный параметр!):")
train_frauds = int(train_customers['fraud_count'].sum())
val_frauds = int(val_customers['fraud_count'].sum())
test_frauds = int(test_customers['fraud_count'].sum())
total_frauds = int(customer_stats['fraud_count'].sum())

print(f"Train: {train_frauds} ({train_frauds/total_frauds*100:.1f}%)")
print(f"Val:   {val_frauds} ({val_frauds/total_frauds*100:.1f}%)")
print(f"Test:  {test_frauds} ({test_frauds/total_frauds*100:.1f}%)")
print(f"Всего: {total_frauds}")

print(f"\nFraud rate (по клиентам - средний):")
print(f"Train: {train_customers['fraud_rate'].mean():.6f}")
print(f"Val:   {val_customers['fraud_rate'].mean():.6f}")
print(f"Test:  {test_customers['fraud_rate'].mean():.6f}")

print(f"\nСреднее число фродов на клиента:")
print(f"Train: {train_customers['fraud_count'].mean():.4f}")
print(f"Val:   {val_customers['fraud_count'].mean():.4f}")
print(f"Test:  {test_customers['fraud_count'].mean():.4f}")

Распределение клиентов по количеству фродов (fraud_count quantile):
fraud_count_quantile
0    3742
Name: count, dtype: int64

Распределение по combined stratification:
Всего уникальных страт: 1

РАЗМЕР РАСЩЕПОВ (по клиентам):
Train клиентов: 2245 (60.0%)
Val клиентов:   748 (20.0%)
Test клиентов:  749 (20.0%)

Пересечения клиентов:
Train ∩ Val:  0 ✓ OK
Train ∩ Test: 0 ✓ OK
Val ∩ Test:   0 ✓ OK

ПРОВЕРКА БАЛАНСА ПО КЛИЕНТАМ:

Общее число фродов (важный параметр!):
Train: 90 (46.6%)
Val:   45 (23.3%)
Test:  58 (30.1%)
Всего: 193

Fraud rate (по клиентам - средний):
Train: 0.004995
Val:   0.007041
Test:  0.005753

Среднее число фродов на клиента:
Train: 0.0401
Val:   0.0602
Test:  0.0774


## 6. Распределение транзакций по расщепам

In [18]:
# Распределяем транзакции по расщепам на основе CustomerId
train_df = df[df['CustomerId'].isin(train_ids)].copy()
val_df = df[df['CustomerId'].isin(val_ids)].copy()
test_df = df[df['CustomerId'].isin(test_ids)].copy()

print(f"\n{'='*60}")
print("ФИНАЛЬНАЯ СТАТИСТИКА")
print(f"{'='*60}")

print(f"\nРазмер по транзакциям:")
print(f"Train: {len(train_df):6d} ({len(train_df)/len(df)*100:5.1f}%)")
print(f"Val:   {len(val_df):6d} ({len(val_df)/len(df)*100:5.1f}%)")
print(f"Test:  {len(test_df):6d} ({len(test_df)/len(df)*100:5.1f}%)")
print(f"Всего: {len(df):6d}")

print(f"\nFraud rate (по транзакциям):")
print(f"Train: {train_df['FraudResult'].mean():.4f} ({train_df['FraudResult'].sum():.0f} фрод)")
print(f"Val:   {val_df['FraudResult'].mean():.4f} ({val_df['FraudResult'].sum():.0f} фрод)")
print(f"Test:  {test_df['FraudResult'].mean():.4f} ({test_df['FraudResult'].sum():.0f} фрод)")
print(f"Исход: {df['FraudResult'].mean():.4f} ({df['FraudResult'].sum():.0f} фрод)")

print(f"\nFraud rate (по клиентам):")
train_fraud_rate_by_customer = train_df.groupby('CustomerId')['FraudResult'].mean().mean()
val_fraud_rate_by_customer = val_df.groupby('CustomerId')['FraudResult'].mean().mean()
test_fraud_rate_by_customer = test_df.groupby('CustomerId')['FraudResult'].mean().mean()

print(f"Train: {train_fraud_rate_by_customer:.4f}")
print(f"Val:   {val_fraud_rate_by_customer:.4f}")
print(f"Test:  {test_fraud_rate_by_customer:.4f}")
print(f"Исход: {df.groupby('CustomerId')['FraudResult'].mean().mean():.4f}")

print(f"\nУникальные клиенты:")
print(f"Train: {train_df['CustomerId'].nunique()}")
print(f"Val:   {val_df['CustomerId'].nunique()}")
print(f"Test:  {test_df['CustomerId'].nunique()}")
print(f"Всего: {df['CustomerId'].nunique()}")


ФИНАЛЬНАЯ СТАТИСТИКА

Размер по транзакциям:
Train:  54459 ( 56.9%)
Val:    21537 ( 22.5%)
Test:   19666 ( 20.6%)
Всего:  95662

Fraud rate (по транзакциям):
Train: 0.0017 (90 фрод)
Val:   0.0021 (45 фрод)
Test:  0.0029 (58 фрод)
Исход: 0.0020 (193 фрод)

Fraud rate (по клиентам):
Train: 0.0050
Val:   0.0070
Test:  0.0058
Исход: 0.0056

Уникальные клиенты:
Train: 2245
Val:   748
Test:  749
Всего: 3742


## 7. Сохранение файлов

In [19]:
# Сохраняем RAW файлы (со всеми колонками)
train_df.to_csv('train_raw.csv', index=False)
val_df.to_csv('val_raw.csv', index=False)
test_df.to_csv('test_raw.csv', index=False)

print(f"\n{'='*60}")
print("✓ RAW ФАЙЛЫ УСПЕШНО СОХРАНЕНЫ!")
print(f"{'='*60}")
print(f"train_raw.csv - {len(train_df)} строк, {train_df.shape[1]} колонок")
print(f"val_raw.csv   - {len(val_df)} строк, {val_df.shape[1]} колонок")
print(f"test_raw.csv  - {len(test_df)} строк, {test_df.shape[1]} колонок")


✓ RAW ФАЙЛЫ УСПЕШНО СОХРАНЕНЫ!
train_raw.csv - 54459 строк, 16 колонок
val_raw.csv   - 21537 строк, 16 колонок
test_raw.csv  - 19666 строк, 16 колонок


## 8. Удаление ненужных колонок и сохранение финальных файлов

In [23]:
# Удаляем ненужные колонки из RAW файлов
cols_to_drop = ['CountQuantile', 'BatchId', 'SubscriptionId', 'AccountId', 'CurrencyCode', 'CountryCode', 'Value']

# Удаляем только существующие колонки
cols_to_drop = [col for col in cols_to_drop if col in train_df.columns]

print(f"Удаляем колонки: {cols_to_drop}")

# Создаём очищенные версии
train_clean = train_df.drop(columns=cols_to_drop).copy()
val_clean = val_df.drop(columns=cols_to_drop).copy()
test_clean = test_df.drop(columns=cols_to_drop).copy()

print(f"Осталось колонок: {train_clean.shape[1]}")
print(f"Новые колонки: {train_clean.columns.tolist()}")

# Сохраняем очищенные файлы (без суффикса raw)
train_clean.to_csv('train.csv', index=False)
val_clean.to_csv('val.csv', index=False)
test_clean.to_csv('test.csv', index=False)

print(f"\n{'='*60}")
print("✓ ОЧИЩЕННЫЕ ФАЙЛЫ УСПЕШНО СОХРАНЕНЫ!")
print(f"{'='*60}")
print(f"train.csv - {len(train_clean)} строк, {train_clean.shape[1]} колонок")
print(f"val.csv   - {len(val_clean)} строк, {val_clean.shape[1]} колонок")
print(f"test.csv  - {len(test_clean)} строк, {test_clean.shape[1]} колонок")

print(f"\n{'='*60}")
print("ИТОГОВАЯ СТАТИСТИКА ОЧИЩЕННЫХ ФАЙЛОВ")
print(f"{'='*60}")

print(f"\nFraud rate (по транзакциям):")
print(f"Train: {train_clean['FraudResult'].mean():.4f}")
print(f"Val:   {val_clean['FraudResult'].mean():.4f}")
print(f"Test:  {test_clean['FraudResult'].mean():.4f}")

Удаляем колонки: ['BatchId', 'SubscriptionId', 'AccountId', 'CurrencyCode', 'CountryCode', 'Value']
Осталось колонок: 10
Новые колонки: ['TransactionId', 'CustomerId', 'ProviderId', 'ProductId', 'ProductCategory', 'ChannelId', 'Amount', 'TransactionStartTime', 'PricingStrategy', 'FraudResult']

✓ ОЧИЩЕННЫЕ ФАЙЛЫ УСПЕШНО СОХРАНЕНЫ!
train.csv - 54459 строк, 10 колонок
val.csv   - 21537 строк, 10 колонок
test.csv  - 19666 строк, 10 колонок

ИТОГОВАЯ СТАТИСТИКА ОЧИЩЕННЫХ ФАЙЛОВ

Fraud rate (по транзакциям):
Train: 0.0017
Val:   0.0021
Test:  0.0029
